Use the rexomni environment

In [1]:
import torch
from PIL import Image
from pathlib import Path
import json
import os, sys
import time
import random
import statistics

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(repo_root)
sys.path.append(os.path.join(repo_root, "src"))

from annotation_methods.io_utils import write_coco_output, retrieve_image_batches

from rex_omni import RexOmniWrapper


In [2]:
# ---- Reusable constants ----
DATA_ROOT = Path("../../Data")

RESULTS_PATH = Path("../../Results/Experiment_1")

DATASET_DICT = {
    "apples": ["good apple", "bad apple"],
    "tomatoes": ["tomato"],
}

In [3]:
rex_backends = ["vllm","transformers"]

def initiate_rex_model(backend:str):
    if backend == "transformers":
        # Initialize the wrapper (model loads internally)
        rex_model = RexOmniWrapper(
            model_path="IDEA-Research/Rex-Omni",   # HF repo
            backend="transformers",                # or "vllm" for high-throughput inference
            device_map="cuda",                     # 🚫 prevents CPU offload
            max_tokens=2048,
            temperature=0.0,
            top_p=0.05,
            top_k=1,
            repetition_penalty=1.05,
        )
    if backend == "vllm":
        # Initialize the wrapper (model loads internally)
        rex_model = RexOmniWrapper(
            model_path="IDEA-Research/Rex-Omni",   # HF repo
            backend="vllm",
            max_tokens=2048,
            temperature=0.0,
            top_p=0.05,
            top_k=1,
            repetition_penalty=1.05,
        )        
    return rex_model

In [4]:
def run_rex_inference_to_coco(
    dataset: str,
    rex_backend: str,
    rex_model: RexOmniWrapper,
    data_root: Path,
    output_path: Path,
    categories: list,
    batch_size: int = 4,
    sample_size: int | None = None,
    model_name: str = "rexomni",
    repeat: int | None = None,
    warmup_steps: int = 1, 
):
    """
    Run REX detection inference on a test split and write COCO-format JSON.
    """

    images_folder = data_root / dataset / "images" / "test"

    images = []
    annotations = []

    img_id = 1
    ann_id = 1
    num_pred_boxes = 0
    total_inference_time_s = 0.0
    first_batch = True
    batch_times_s: list[float] = []

    if repeat is not None:
        model_name = f"{model_name}_{rex_backend}_r{repeat}"
    else:
        model_name = f"{model_name}_{rex_backend}"

    # Run inference
    for batch_images, names in retrieve_image_batches(
        images_folder, batch_size=batch_size, sample_size=sample_size
    ):
        if first_batch:
            # Warmup on first batch (not timed, no results stored)
            for _ in range(warmup_steps):
                _ = rex_model.inference(
                    images=batch_images, task="detection", categories=categories
                )
            first_batch = False

        # Timed inference
        torch.cuda.synchronize()
        start = time.perf_counter()

        results = rex_model.inference(
            images=batch_images,
            task="detection",
            categories=categories,
        )

        torch.cuda.synchronize()
        end = time.perf_counter()

        batch_time = end - start
        total_inference_time_s += batch_time
        batch_times_s.append(batch_time)

        # COCO conversion
        for name, res, im in zip(names, results, batch_images):
            if not res.get("success", False):
                continue

            w, h = res["image_size"]
            current_img_id = img_id
            images.append(
                {
                    "id": current_img_id,
                    "file_name": f"images/test/{name}",
                }
            )
            img_id += 1

            preds = res.get("extracted_predictions", {})
            for label, objs in preds.items():
                if label not in categories:
                    continue
                category_id = categories.index(label)

                for obj in objs:
                    if obj.get("type") != "box":
                        continue

                    x0, y0, x1, y1 = obj["coords"]

                    x0 = max(0.0, min(float(x0), float(w)))
                    x1 = max(0.0, min(float(x1), float(w)))
                    y0 = max(0.0, min(float(y0), float(h)))
                    y1 = max(0.0, min(float(y1), float(h)))

                    bbox_w = max(0.0, x1 - x0)
                    bbox_h = max(0.0, y1 - y0)
                    area = bbox_w * bbox_h

                    if bbox_w <= 0 or bbox_h <= 0:
                        continue

                    annotations.append(
                        {
                            "id": ann_id,
                            "image_id": current_img_id,
                            "category_id": category_id,
                            "bbox": [x0, y0, bbox_w, bbox_h],
                            "area": area,
                            "iscrowd": 0,
                        }
                    )
                    ann_id += 1
                    num_pred_boxes += 1

    num_images = len(images)

    # ------------------------------------------------------------------
    # Write COCO JSON and return path
    # ------------------------------------------------------------------
    out_file = write_coco_output(
        images_folder=str(images_folder),
        model_name=model_name,
        categories_list=categories,
        images=images,
        annotations=annotations,
        num_images=num_images,
        output_path=output_path,
        num_initial_bbox=0,  # zero-shot has no initial boxes
        num_pred_boxes=num_pred_boxes,
        total_inference_time_s=total_inference_time_s,
        machine_training_time_s=0.0,  # zero-shot requires no training time
        batch_times_s=batch_times_s,
        batch_size=batch_size,  
    )

    print(f"Wrote COCO-format JSON to {out_file}")
    return out_file


## Inference Runs

In [5]:
for backend in rex_backends:
    rex_model = initiate_rex_model(backend)
    dataset = 'apples'
    for repeat in range(0,3):
        run_rex_inference_to_coco(
            dataset=dataset,
            rex_model=rex_model,
            rex_backend=backend,
            data_root=DATA_ROOT,
            output_path=RESULTS_PATH,
            categories=DATASET_DICT[dataset],
            batch_size=8,
            warmup_steps=3,
            repeat=repeat,
        )

    # cleanup after all datasets for this backend
    try:
        if hasattr(rex_model, "close"):
            rex_model.close()
        elif hasattr(rex_model, "shutdown"):
            rex_model.shutdown()
    except Exception:
        pass

    import gc, torch
    del rex_model
    gc.collect()
    torch.cuda.empty_cache()

Initializing vllm backend...


/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 01-02 21:07:41 [__init__.py:244] Automatically detected platform cuda.


2026-01-02 21:07:47,982	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 01-02 21:08:05 [config.py:823] This model supports multiple tasks: {'embed', 'generate', 'reward', 'score', 'classify'}. Defaulting to 'generate'.
INFO 01-02 21:08:07 [config.py:3268] Downcasting torch.float32 to torch.bfloat16.
INFO 01-02 21:08:14 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 01-02 21:08:17 [tokenizer.py:262] Using a slow tokenizer. This might cause a significant slowdown. Consider using a fast tokenizer instead.
INFO 01-02 21:08:18 [core.py:455] Waiting for init message from front-end.
INFO 01-02 21:08:18 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='IDEA-Research/Rex-Omni', speculative_config=None, tokenizer='IDEA-Research/Rex-Omni', skip_tokenizer_init=False, tokenizer_mode=slow, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, di

...

Wrote COCO-format JSON to ../../Results/Experiment_1/apples_test_rexomni_vllm_r2_predictions.json
Initializing transformers backend...


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.23it/s]


Found 31 image files


/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.05` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/home/warredv/minico

Wrote COCO-format JSON to ../../Results/Experiment_1/apples_test_rexomni_transformers_r0_predictions.json
Found 31 image files
Wrote COCO-format JSON to ../../Results/Experiment_1/apples_test_rexomni_transformers_r1_predictions.json
Found 31 image files
Wrote COCO-format JSON to ../../Results/Experiment_1/apples_test_rexomni_transformers_r2_predictions.json


In [5]:
for backend in rex_backends:
    rex_model = initiate_rex_model(backend)
    dataset = 'tomatoes'
    for repeat in range(0,3):
        run_rex_inference_to_coco(
            dataset=dataset,
            rex_model=rex_model,
            rex_backend=backend,
            data_root=DATA_ROOT,
            output_path=RESULTS_PATH,
            categories=DATASET_DICT[dataset],
            batch_size=8,
            warmup_steps=3,
            repeat=repeat,
        )

    # cleanup after all datasets for this backend
    try:
        if hasattr(rex_model, "close"):
            rex_model.close()
        elif hasattr(rex_model, "shutdown"):
            rex_model.shutdown()
    except Exception:
        pass

    import gc, torch
    del rex_model
    gc.collect()
    torch.cuda.empty_cache()

Initializing vllm backend...


/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 01-02 21:15:52 [__init__.py:244] Automatically detected platform cuda.


2026-01-02 21:15:58,771	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 01-02 21:16:16 [config.py:823] This model supports multiple tasks: {'embed', 'score', 'generate', 'classify', 'reward'}. Defaulting to 'generate'.
INFO 01-02 21:16:18 [config.py:3268] Downcasting torch.float32 to torch.bfloat16.
INFO 01-02 21:16:25 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 01-02 21:16:28 [tokenizer.py:262] Using a slow tokenizer. This might cause a significant slowdown. Consider using a fast tokenizer instead.
INFO 01-02 21:16:28 [core.py:455] Waiting for init message from front-end.
INFO 01-02 21:16:28 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='IDEA-Research/Rex-Omni', speculative_config=None, tokenizer='IDEA-Research/Rex-Omni', skip_tokenizer_init=False, tokenizer_mode=slow, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, di

...

Wrote COCO-format JSON to ../../Results/Experiment_1/tomatoes_test_rexomni_vllm_r2_predictions.json
Initializing transformers backend...


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.23it/s]


Found 662 image files


/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.05` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/home/warredv/minico

Wrote COCO-format JSON to ../../Results/Experiment_1/tomatoes_test_rexomni_transformers_r0_predictions.json
Found 662 image files
Wrote COCO-format JSON to ../../Results/Experiment_1/tomatoes_test_rexomni_transformers_r1_predictions.json
Found 662 image files
Wrote COCO-format JSON to ../../Results/Experiment_1/tomatoes_test_rexomni_transformers_r2_predictions.json


old tries

In [ ]:
# run inference on all datasets and models
for dataset in DATASET_DICT.keys():
    for backend in rex_backends:
            run_rex_inference_to_coco(
                dataset=dataset, 
                data_root=DATA_ROOT,
                output_path=RESULTS_PATH,
                categories=DATASET_DICT[dataset],
                rex_backend=backend,
                batch_size=4,
                warmup_steps=3,
            )

## OOM tests

vllm with batch size 8 both sits really close to max VRAM so batch size of 8 is justified

In [5]:
dataset = "tomatoes"
# rex_backend = "transformers"
rex_backend = "vllm"
categories = DATASET_DICT.get(dataset)
rex_model = initiate_rex_model(rex_backend)

run_rex_inference_to_coco(dataset=dataset, 
                          rex_backend=rex_backend,
                          rex_model=rex_model,
                          data_root=DATA_ROOT,
                          output_path=RESULTS_PATH,
                          categories=categories,
                          batch_size=8,
                          warmup_steps=3,
                          sample_size=64
                          )

Initializing vllm backend...


/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 01-02 17:50:46 [__init__.py:244] Automatically detected platform cuda.


2026-01-02 17:50:52,204	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 01-02 17:51:09 [config.py:823] This model supports multiple tasks: {'reward', 'classify', 'score', 'embed', 'generate'}. Defaulting to 'generate'.
INFO 01-02 17:51:11 [config.py:3268] Downcasting torch.float32 to torch.bfloat16.
INFO 01-02 17:51:18 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 01-02 17:51:21 [tokenizer.py:262] Using a slow tokenizer. This might cause a significant slowdown. Consider using a fast tokenizer instead.
INFO 01-02 17:51:22 [core.py:455] Waiting for init message from front-end.
INFO 01-02 17:51:22 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='IDEA-Research/Rex-Omni', speculative_config=None, tokenizer='IDEA-Research/Rex-Omni', skip_tokenizer_init=False, tokenizer_mode=slow, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, di

...

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Found 662 image files


Processed prompts: 100%|██████████| 8/8 [00:01<00:00,  7.68it/s, est. speed input: 815.52 toks/s, output: 233.41 toks/s]


Wrote COCO-format JSON to ../../Results/Experiment_1/tomatoes_test_rexomni_vllm_predictions.json


'../../Results/Experiment_1/tomatoes_test_rexomni_vllm_predictions.json'

In [ ]:
dataset = "apples"
# rex_backend = "transformers"
rex_backend = "vllm"
categories = DATASET_DICT.get(dataset)
rex_model = initiate_rex_model(rex_backend)

run_rex_inference_to_coco(dataset=dataset, 
                          rex_backend=rex_backend,
                          rex_model=rex_model,
                          data_root=DATA_ROOT,
                          output_path=RESULTS_PATH,
                          categories=categories,
                          batch_size=8,
                          warmup_steps=3,
                          )

Initializing vllm backend...


INFO 01-02 18:10:27 [config.py:823] This model supports multiple tasks: {'classify', 'embed', 'score', 'reward', 'generate'}. Defaulting to 'generate'.
INFO 01-02 18:10:27 [config.py:3268] Downcasting torch.float32 to torch.bfloat16.
INFO 01-02 18:10:27 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 01-02 18:10:28 [tokenizer.py:262] Using a slow tokenizer. This might cause a significant slowdown. Consider using a fast tokenizer instead.
WARNING 01-02 18:10:28 [utils.py:2597] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reason: CUDA is initialized
WARNING 01-02 18:10:29 [env_override.py:17] NCCL_CUMEM_ENABLE is set to 0, skipping override. This may increase memory overhead with cudagraph+allreduce: https://github.com/NVIDIA/nccl/issues/1234
INFO 01-02 18:10:31 [__init__.py:244] Autom

Process EngineCore_0:
Traceback (most recent call last):
  File "/home/warredv/miniconda3/envs/rexomni/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/warredv/miniconda3/envs/rexomni/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 519, in run_engine_core
    raise e
  File "/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 506, in run_engine_core
    engine_core = EngineCoreProc(*args, **kwargs)
  File "/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 390, in __init__
    super().__init__(vllm_config, executor_class, log_stats,
  File "/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 76, in __init__
    self.model_executor = executor_cl

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [5]:
dataset = "apples"
rex_backend = "transformers"
# rex_backend = "vllm"
categories = DATASET_DICT.get(dataset)
rex_model = initiate_rex_model(rex_backend)

run_rex_inference_to_coco(dataset=dataset, 
                          rex_backend=rex_backend,
                          rex_model=rex_model,
                          data_root=DATA_ROOT,
                          output_path=RESULTS_PATH,
                          categories=categories,
                          batch_size=8,
                          warmup_steps=3,
                          )

Initializing transformers backend...


/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.13it/s]


Found 31 image files


/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.05` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/home/warredv/minico

Wrote COCO-format JSON to ../../Results/Experiment_1/apples_test_rexomni_transformers_predictions.json


'../../Results/Experiment_1/apples_test_rexomni_transformers_predictions.json'

In [6]:
dataset = "tomatoes"
rex_backend = "transformers"
# rex_backend = "vllm"
categories = DATASET_DICT.get(dataset)
# rex_model = initiate_rex_model(rex_backend)

run_rex_inference_to_coco(dataset=dataset, 
                          rex_backend=rex_backend,
                          rex_model=rex_model,
                          data_root=DATA_ROOT,
                          output_path=RESULTS_PATH,
                          categories=categories,
                          batch_size=8,
                          warmup_steps=3,
                          )

Found 662 image files


KeyboardInterrupt: 

## Debugging

In [4]:
import time
import gc
from pathlib import Path
from typing import List, Optional, Tuple
import torch

def run_rex_inference_to_coco_debug(
    dataset: str,
    rex_backend: str,
    rex_model,  # RexOmniWrapper
    data_root: Path,
    output_path: Path,
    categories: List[str],
    batch_size: int = 4,
    sample_size: Optional[int] = None,
    model_name: str = "rexomni",
    warmup_steps: int = 1,
    debug_max_batches: Optional[int] = None,  # limit printed batches, None = all
) -> Path:
    """
    Debug version of run_rex_inference_to_coco with detailed timing and statistics.

    - Logs per-batch timing, per-image timing, image sizes.
    - Logs per-image number of predicted boxes.
    - Prints summary stats at the end.

    Parameters
    ----------
    dataset : str
        Dataset name, used to look up categories and paths.
    rex_backend : str
        Backend identifier for logging.
    rex_model : RexOmniWrapper
        Initialized REX model wrapper.
    data_root : Path
        Root path of all datasets (parent of `<dataset>/images/test`).
    output_path : Path
        Directory where the COCO JSON and logs will be written.
    categories : list[str]
        List of category names.
    batch_size : int, optional
        Batch size for retrieve_image_batches.
    sample_size : int | None, optional
        Sample size for retrieve_image_batches (pass None for all images).
    model_name : str, optional
        Name to pass to write_coco_output. (is updated with rex_backend)
    warmup_steps : int, optional
        How many batches to process (not timed) for warmup.
    debug_max_batches : int | None, optional
        If not None, only logs details for the first N batches.

    Returns
    -------
    out_file : Path
        Path returned by write_coco_output.
    """

    # Resolve paths
    images_folder = data_root / dataset / "images" / "test"

    print("=" * 80)
    print(f"[DEBUG] Starting inference for dataset='{dataset}', backend='{rex_backend}'")
    print(f"[DEBUG] images_folder = {images_folder}")
    print(f"[DEBUG] num_categories = {len(categories)}")
    print(f"[DEBUG] categories = {categories}")
    print(f"[DEBUG] batch_size = {batch_size}, sample_size = {sample_size}")
    print(f"[DEBUG] warmup_steps = {warmup_steps}")
    print("=" * 80)

    images = []
    annotations = []

    img_id = 1
    ann_id = 1
    num_pred_boxes = 0
    total_inference_time_s = 0.0
    first_batch = True

    model_name = f"{model_name}_{rex_backend}"

    # Debug accumulators
    total_images_seen = 0
    total_boxes_seen = 0
    batch_idx = 0

    # ------------------------------------------------------------------
    # Main batching loop
    # ------------------------------------------------------------------
    for batch_images, names in retrieve_image_batches(
        images_folder, batch_size=batch_size, sample_size=sample_size
    ):
        n_batch_images = len(batch_images)
        if n_batch_images == 0:
            continue

        # Print basic batch info
        print(f"\n[DEBUG] Dataset='{dataset}' backend='{rex_backend}' batch={batch_idx}")
        print(f"[DEBUG]   batch_size_actual = {n_batch_images}")

        # Log some image sizes (up to 3 per batch)
        sample_sizes = [im.size for im in batch_images[:3]]
        print(f"[DEBUG]   sample image sizes (W,H) = {sample_sizes}")

        # Warmup on first batch (not timed, no results stored)
        if first_batch:
            print(f"[DEBUG]   running warmup for {warmup_steps} step(s) on first batch")
            for _ in range(warmup_steps):
                _ = rex_model.inference(
                    images=batch_images,
                    task="detection",
                    categories=categories,
                )
            first_batch = False

        # Timed inference
        torch.cuda.synchronize()
        start = time.perf_counter()

        results = rex_model.inference(
            images=batch_images,
            task="detection",
            categories=categories,
        )

        torch.cuda.synchronize()
        end = time.perf_counter()

        batch_time = end - start
        per_image_time = batch_time / n_batch_images
        total_inference_time_s += batch_time

        print(
            f"[DEBUG]   inference time = {batch_time:.3f}s "
            f"({per_image_time:.3f}s / image)"
        )

        # COCO conversion + per-image stats
        batch_boxes = 0
        for name, res, im in zip(names, results, batch_images):
            # Skip if inference failed
            if not res.get("success", False):
                print(f"[DEBUG]   WARNING: inference failed for image '{name}'")
                continue

            # Original image size: (width, height)
            w, h = res["image_size"]

            # Register image entry
            current_img_id = img_id
            images.append(
                {
                    "id": current_img_id,
                    "file_name": f"images/test/{name}",
                }
            )
            img_id += 1
            total_images_seen += 1

            preds = res.get("extracted_predictions", {})
            image_box_count = 0

            for label, objs in preds.items():
                if label not in categories:
                    continue
                category_id = categories.index(label)

                for obj in objs:
                    if obj.get("type") != "box":
                        continue

                    x0, y0, x1, y1 = obj["coords"]

                    # Clamp coordinates to image bounds
                    x0 = max(0.0, min(float(x0), float(w)))
                    x1 = max(0.0, min(float(x1), float(w)))
                    y0 = max(0.0, min(float(y0), float(h)))
                    y1 = max(0.0, min(float(y1), float(h)))

                    # Convert [x0, y0, x1, y1] -> [x, y, width, height]
                    bbox_w = max(0.0, x1 - x0)
                    bbox_h = max(0.0, y1 - y0)
                    area = bbox_w * bbox_h

                    # Skip degenerate boxes
                    if bbox_w <= 0 or bbox_h <= 0:
                        continue

                    annotations.append(
                        {
                            "id": ann_id,
                            "image_id": current_img_id,
                            "category_id": category_id,
                            "bbox": [x0, y0, bbox_w, bbox_h],
                            "area": area,
                            "iscrowd": 0,
                        }
                    )
                    ann_id += 1
                    num_pred_boxes += 1
                    batch_boxes += 1
                    image_box_count += 1

            total_boxes_seen += image_box_count
            print(
                f"[DEBUG]   image='{name}' size=({w},{h}) "
                f"pred_boxes={image_box_count}"
            )

        print(
            f"[DEBUG]   batch={batch_idx} total_boxes_in_batch={batch_boxes} "
            f"(running total_boxes={total_boxes_seen})"
        )

        batch_idx += 1
        if debug_max_batches is not None and batch_idx >= debug_max_batches:
            print(
                f"[DEBUG]   Reached debug_max_batches={debug_max_batches}, "
                f"continuing without further detailed per-batch logs."
            )
            # After this, we stop printing but continue processing
            debug_max_batches = None  # disable limitation, but no more detailed logs

    # ------------------------------------------------------------------
    # Write COCO JSON and print summary
    # ------------------------------------------------------------------
    num_images = len(images)

    if num_images > 0:
        avg_time_per_image = total_inference_time_s / num_images
    else:
        avg_time_per_image = 0.0

    print("\n" + "=" * 80)
    print(f"[DEBUG] Finished dataset='{dataset}', backend='{rex_backend}'")
    print(f"[DEBUG]   num_images          = {num_images}")
    print(f"[DEBUG]   num_pred_boxes      = {num_pred_boxes}")
    print(f"[DEBUG]   total_infer_time_s  = {total_inference_time_s:.3f}s")
    print(f"[DEBUG]   avg_time_per_image  = {avg_time_per_image:.3f}s")
    print("=" * 80)

    out_file = write_coco_output(
        images_folder=str(images_folder),
        model_name=model_name,
        categories_list=categories,
        images=images,
        annotations=annotations,
        num_images=num_images,
        output_path=output_path,
        num_initial_bbox=0,  # zero-shot has no initial boxes
        num_pred_boxes=num_pred_boxes,
        total_inference_time_s=total_inference_time_s,
        machine_training_time_s=0.0,  # zero-shot requires no training time
    )

    print(f"[DEBUG] Wrote COCO-format JSON to {out_file}")
    print("=" * 80 + "\n")

    return out_file


In [5]:
for backend in rex_backends:
    rex_model = initiate_rex_model(backend)

    for dataset in DATASET_DICT.keys():
        run_rex_inference_to_coco_debug(
            dataset=dataset,
            rex_model=rex_model,
            rex_backend=backend,
            data_root=DATA_ROOT,
            output_path=RESULTS_PATH,
            categories=DATASET_DICT[dataset],
            batch_size=4,
            warmup_steps=3,
            sample_size=8,
            debug_max_batches=5,  # or None to log all
        )

    # cleanup as before
    try:
        if hasattr(rex_model, "close"):
            rex_model.close()
        elif hasattr(rex_model, "shutdown"):
            rex_model.shutdown()
    except Exception:
        pass

    del rex_model
    gc.collect()
    torch.cuda.empty_cache()


Initializing vllm backend...


/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 01-02 15:45:24 [__init__.py:244] Automatically detected platform cuda.


2026-01-02 15:45:30,608	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 01-02 15:45:48 [config.py:823] This model supports multiple tasks: {'generate', 'reward', 'embed', 'classify', 'score'}. Defaulting to 'generate'.
INFO 01-02 15:45:49 [config.py:3268] Downcasting torch.float32 to torch.bfloat16.
INFO 01-02 15:45:57 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 01-02 15:46:00 [tokenizer.py:262] Using a slow tokenizer. This might cause a significant slowdown. Consider using a fast tokenizer instead.
INFO 01-02 15:46:00 [core.py:455] Waiting for init message from front-end.
INFO 01-02 15:46:00 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='IDEA-Research/Rex-Omni', speculative_config=None, tokenizer='IDEA-Research/Rex-Omni', skip_tokenizer_init=False, tokenizer_mode=slow, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, di

...

[DEBUG]   inference time = 22.265s (5.566s / image)
[DEBUG]   image='2023_08_21__12_58_29_728314000___02Z8R__02Z8W_bunch_5.png' size=(115,180) pred_boxes=282
[DEBUG]   image='2023_08_25__18_56_10_490538000___02Z8R__02Z8W_bunch_4.png' size=(141,286) pred_boxes=4
[DEBUG]   image='2023_11_22__15_09_25_747000000___04ZXJ__04ZXK_bunch_7.png' size=(124,134) pred_boxes=3
[DEBUG]   image='2023_11_22__15_21_47_350000000___04ZXJ__04ZXK_bunch_2.png' size=(152,156) pred_boxes=3
[DEBUG]   batch=1 total_boxes_in_batch=292 (running total_boxes=317)

[DEBUG] Finished dataset='tomatoes', backend='vllm'
[DEBUG]   num_images          = 8
[DEBUG]   num_pred_boxes      = 317
[DEBUG]   total_infer_time_s  = 23.203s
[DEBUG]   avg_time_per_image  = 2.900s
[DEBUG] Wrote COCO-format JSON to ../../Results/Experiment_1/tomatoes_test_rexomni_vllm_predictions.json

Initializing transformers backend...


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.24it/s]


[DEBUG] Starting inference for dataset='apples', backend='transformers'
[DEBUG] images_folder = ../../Data/apples/images/test
[DEBUG] num_categories = 2
[DEBUG] categories = ['good apple', 'bad apple']
[DEBUG] batch_size = 4, sample_size = 8
[DEBUG] warmup_steps = 3
Found 31 image files

[DEBUG] Dataset='apples' backend='transformers' batch=0
[DEBUG]   batch_size_actual = 4
[DEBUG]   sample image sizes (W,H) = [(1959, 1574), (1959, 1574), (1959, 1574)]
[DEBUG]   running warmup for 3 step(s) on first batch


/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.05` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/home/warredv/minico

[DEBUG]   inference time = 8.695s (2.174s / image)
[DEBUG]   image='IMG_0386.png' size=(1959,1574) pred_boxes=13
[DEBUG]   image='IMG_0335.png' size=(1959,1574) pred_boxes=15
[DEBUG]   image='IMG_0298.png' size=(1959,1574) pred_boxes=21
[DEBUG]   image='IMG_0365.png' size=(1959,1574) pred_boxes=13
[DEBUG]   batch=0 total_boxes_in_batch=62 (running total_boxes=62)

[DEBUG] Dataset='apples' backend='transformers' batch=1
[DEBUG]   batch_size_actual = 4
[DEBUG]   sample image sizes (W,H) = [(1959, 1574), (1959, 1574), (1959, 1574)]
[DEBUG]   inference time = 7.115s (1.779s / image)
[DEBUG]   image='IMG_0429.png' size=(1959,1574) pred_boxes=14
[DEBUG]   image='IMG_0405.png' size=(1959,1574) pred_boxes=12
[DEBUG]   image='IMG_0394.png' size=(1959,1574) pred_boxes=14
[DEBUG]   image='IMG_0388.png' size=(1959,1574) pred_boxes=12
[DEBUG]   batch=1 total_boxes_in_batch=52 (running total_boxes=114)

[DEBUG] Finished dataset='apples', backend='transformers'
[DEBUG]   num_images          = 8
[DEBU